# 2 - Prepare Isofox RNA Gene Sets

**Purpose**: Compute the T-cell immune gene set score from Isofox gene expression, plus a set of independent, published RNA comparator signatures (effector, PD-L1, exhaustion, and T-cell-inflamed GEP panels, and a second T-cell presence signature), all for comparison against the CIDER T-cell score in the supplemental gene-set validation figure (`1_figures/21_tcell_geneset_panel.ipynb`) and the DNA-vs-RNA figure (`1_figures/19_dna_vs_rna_response.ipynb`). This project is T-cell-focused (see `helpers/config.r`'s `GP_LEVELS`); B-cell/Lymphocyte gene sets were removed from scope. 6 gene sets are computed in total: the primary T-cell signature, T-cell (HCL), CD8 Effector (tGE8), PD-L1, T-cell-inflamed GEP, and the checkpoint/exhaustion panel.

Every panel below other than the primary T-cell signature and the
T-cell-inflamed GEP was chosen because its exact gene list could be verified
from a primary source (the original paper's own supplementary table)
rather than a secondhand reconstruction. The T-cell-inflamed GEP (Ayers et
al. 2017, below) is the one exception -- its gene list is a secondary-source
reconstruction, included per explicit direction despite the JCI paper's own
supplementary table being unreachable as extractable text in every fetch
attempted. Two other well-known panels from the literature (Danaher et al.
2017, Bindea et al. 2013) remain deliberately NOT implemented, for the same
reason -- see `fig_references/tcell_signatures_literature.md` for the full
verification history of all three.

Gene panels:
- **T-cell**: CD3D, CD3E, CD3G -- Braun, D.A. et al. "Progressive immune dysfunction with advancing disease stage in renal cell carcinoma." *Cancer Cell* 39(5):632-648.e8 (2021). PMID 33711273.
- **CD8 Effector (tGE8)**: CD8A, GZMA, GZMB, IFNG, CXCL9, CXCL10, PRF1, TBX21 — T-effector/IFN-γ signature used across Roche/Genentech atezolizumab trials (e.g. Mariathasan et al. 2018 *Nature*; Rosenberg et al. 2016 *Lancet*, IMvigor210). This is a distinct, additional comparator — it does NOT replace the T-cell panel above, which remains the primary RNA validation score for the CIDER T-cell Score (see `4_cider_validation.ipynb`). It is used only in supplemental DNA-vs-RNA limitation analyses.
- **PD-L1**: CD274 — bulk RNA expression of the gene encoding PD-L1, the protein measured by IHC (Tumor Proportion Score / Combined Positive Score) as the standard clinical predictive biomarker for checkpoint inhibitor therapy. A single gene, not a composite signature; included as a third RNA comparator in `19_dna_vs_rna_response.ipynb` alongside the T-cell and CD8 Effector scores above, and as an Exhausted-category comparator in `21_tcell_geneset_panel.ipynb`. Note this is a transcript-level proxy, not the IHC assay itself, and PD-L1 expression is partly induced by IFN-γ from infiltrating T cells — so it isn't expected to be fully independent of the other metrics.
- **T-cell (HCL)**: CD3D, CD3E, CD3G, CD2, TRAC — an independent pan-T-cell marker panel from the Human Cell Landscape single-cell atlas (Han, X., Zhou, Z., Fei, L. et al. "Construction of a human cell landscape at single-cell level." *Nature* 581, 303–309, 2020). A second, independently-sourced T-cell presence signature, used as a fourth RNA comparator in `19_dna_vs_rna_response.ipynb` alongside the three above — not a replacement for the primary T-cell panel.
- **Checkpoint/exhaustion panel**: PDCD1, CTLA4, LAG3, HAVCR2, TIGIT — the canonical co-inhibitory-receptor panel used throughout the T-cell exhaustion literature (e.g. Thommen et al. 2018 *Nat Med* 24:994-1004; Tirosh et al. 2016 *Science* 352:189-196). Not a single paper's proprietary gene list — this exact 5-gene combination is reproduced consistently across many papers, so it carries materially lower verification risk than a single study's full supplementary signature.
- **T-cell-inflamed GEP (Ayers GEPS)**: PSMB10, HLA-DQA1, HLA-DRB1, CMKLR1, HLA-E, NKG7, CD8A, CCL5, CXCL9, CD27, CXCR6, IDO1, STAT1, CD274, CD276, LAG3, PDCD1LG2, TIGIT — Ayers et al. 2017 *J Clin Invest* 127(8):2930-2940, the 18-gene pan-tumor T-cell-inflamed gene expression profile (also commercialized as the Tumor Inflammation Signature, TIS) shown to predict clinical response to anti-PD-1 therapy. **Verification caveat**: unlike every other panel above, this exact 18-gene list is a secondary-source reconstruction (two independent citations agree, including the related Danaher et al. 2018 TIS re-description) rather than one confirmed directly against the original paper's own supplementary table, which renders as an image in every fetch attempted. Included per explicit direction; see `fig_references/tcell_signatures_literature.md` for the full verification history.

Aggregation:
- T-cell score: **mean** raw AdjTPM across panel genes (`tcell_tpm`). Downstream notebooks apply their own transform (e.g. a log10(x+1) axis) at analysis/plot time rather than consuming a pre-transformed column.
- CD8 Effector score: **mean** raw AdjTPM across panel genes (`rna_teff_score`) — same aggregation as the T-cell score above, so the two RNA metrics sit on the same average-expression scale and are directly comparable in `19_dna_vs_rna_response.ipynb`.
- PD-L1 score: raw AdjTPM for CD274 (`rna_pdl1_score`) — computed via the same `compute_panel()` mean aggregation as the other panels for consistency, though with a single gene the mean is just that gene's own value.
- T-cell (HCL) score: **mean** raw AdjTPM across panel genes (`rna_tcell_hcl_score`) — same aggregation as the T-cell score above, on the same average-expression scale, for direct comparison.
- Exhaustion score: **mean** raw AdjTPM across panel genes (`rna_exhaustion_score`) — same aggregation convention as every panel above.
- T-cell-inflamed GEP score: **mean** raw AdjTPM across panel genes (`rna_ayers_geps_score`) — same aggregation convention as every panel above.

| | |
|---|---|
| **Inputs** | `ISO_DIR/isf.gene_data.csv` |
| **Outputs** | `CIDER_PROJ_DIR/isofox_genesets_ready_log2.csv` |
| **Run before** | `3_meta.ipynb` |
| **See also** | `fig_references/tcell_signatures_literature.md` (full verification history for every panel, including those NOT implemented here) |


In [ ]:
source('../helpers/map.r')
source('../helpers/shortcuts.r')
source('../helpers/config.r')

assert_files_exist(
  "Isofox gene data" = paste0(ISO_DIR, 'isf.gene_data.csv')
)

# 1 - Define gene panels

In [ ]:
# Primary T-cell signature -- Braun, D.A. et al. "Progressive immune
# dysfunction with advancing disease stage in renal cell carcinoma."
# Cancer Cell 39(5):632-648.e8 (2021). PMID 33711273.
gene_panels <- list(
  T_Cell = c('CD3D', 'CD3E', 'CD3G')
)

# Effector T-cell (tGE8) signature — 8-gene T-effector/IFN-gamma signature
# used across Roche/Genentech atezolizumab trials (e.g. Mariathasan et al.
# 2018 Nature; Rosenberg et al. 2016 Lancet, IMvigor210) as an RNA-based
# T-cell-effector/cytotoxicity score — used here as an orthogonal RNA
# comparator for the CIDER T-cell score. This is additional/separate from
# gene_panels above, NOT a replacement — it feeds only supplemental
# DNA-vs-RNA limitation analyses. Aggregated the same way as the T-cell
# panel above (mean raw AdjTPM), so the two RNA metrics are on the same
# average-expression scale and directly comparable.
effector_panels <- list(
  Teff_CD8 = c('CD8A', 'GZMA', 'GZMB', 'IFNG', 'CXCL9', 'CXCL10', 'PRF1', 'TBX21')
)

# PD-L1 (CD274) bulk RNA expression -- the transcript-level proxy for the
# IHC-based PD-L1 Tumor Proportion Score/Combined Positive Score used
# clinically as a predictive biomarker for checkpoint inhibitor therapy.
# Single gene, not a composite signature -- kept as a one-gene "panel" only
# for consistency with compute_panel()'s interface below, same as the two
# panels above. Used as a third RNA comparator in 19_dna_vs_rna_response.ipynb
# and as an Exhausted-category comparator in 21_tcell_geneset_panel.ipynb.
pdl1_panel <- list(
  PD_L1 = c('CD274')
)

# T-cell (HCL) signature -- a second, independently-sourced pan-T-cell
# marker panel from the Human Cell Landscape single-cell atlas (Han, X.,
# Zhou, Z., Fei, L. et al. Construction of a human cell landscape at
# single-cell level. Nature 581, 303-309, 2020). Additional/separate from
# gene_panels above, NOT a replacement -- used as a fourth RNA comparator
# in 19_dna_vs_rna_response.ipynb. Aggregated the same way (mean raw
# AdjTPM) as the other panels, so all RNA metrics stay on the same
# average-expression scale.
hcl_tcell_panel <- list(
  HCL_TCell = c('CD3D', 'CD3E', 'CD3G', 'CD2', 'TRAC')
)

# Canonical checkpoint/exhaustion panel -- not a single paper's proprietary
# gene list; this exact 5-gene combination recurs across the T-cell
# exhaustion literature (e.g. Thommen et al. 2018 Nat Med 24:994-1004;
# Tirosh et al. 2016 Science 352:189-196), which is why it carries lower
# verification risk than a single study's own supplementary signature.
exhaustion_panel <- list(
  Exhaustion = c('PDCD1', 'CTLA4', 'LAG3', 'HAVCR2', 'TIGIT')
)

# T-cell-inflamed GEP / Tumor Inflammation Signature (TIS), 18 genes -- Ayers
# et al. 2017, J Clin Invest 127(8):2930-2940 (PMID 28650338). UNLIKE every
# other panel in this notebook, this gene list is a SECONDARY-SOURCE
# RECONSTRUCTION, not verified against the original paper's own supplementary
# table: the JCI table renders as an image in every fetch attempted (see
# fig_references/tcell_signatures_literature.md for the full history), so
# this list is assembled from multiple independent secondary citations
# (including the related Danaher et al. 2018 TIS re-description) that agree
# with each other, but could not be cross-checked against the primary
# source. Included anyway per explicit direction; flagged here and in the
# literature-review doc so this lower verification confidence is visible at
# the point of use, not just in a separate reference file. Aggregated the
# same way as every panel above except CYT (mean raw AdjTPM).
ayers_geps_panel <- list(
  Ayers_GEPS = c('PSMB10', 'HLA-DQA1', 'HLA-DRB1', 'CMKLR1', 'HLA-E', 'NKG7',
                 'CD8A', 'CCL5', 'CXCL9', 'CD27', 'CXCR6', 'IDO1', 'STAT1',
                 'CD274', 'CD276', 'LAG3', 'PDCD1LG2', 'TIGIT')
)


# 2 - Load and transform Isofox expression

In [ ]:
# Read per-sample gene expression for panel genes only, pivot wide.
# Filtering to panel genes before pivot avoids loading all 60,000+ genes into memory.
panel_genes <- unique(unlist(c(
  gene_panels, effector_panels, pdl1_panel, hcl_tcell_panel,
  exhaustion_panel, ayers_geps_panel
)))

isofox <- fread(paste0(ISO_DIR, 'isf.gene_data.csv'),
         select = c('sampleId', 'gene', 'AdjTPM')) |>
  filter(gene %in% panel_genes)

message('Samples: ', n_distinct(isofox$sampleId),
        ', Panel genes found: ', n_distinct(isofox$gene), ' of ', length(panel_genes))

# Raw AdjTPM wide table — for mean-TPM gene set scores
isofox_wide_tpm <- isofox |>
  pivot_wider(names_from = gene, values_from = AdjTPM)

# 3 - Compute gene set scores

In [ ]:
# Each score = mean expression across panel genes present in the data, on
# the raw AdjTPM scale — used for both the T-cell signature and the CD8
# Effector (tGE8) score below, so the two RNA metrics sit on the same
# average-expression scale and are directly comparable.
compute_panel <- function(panel_name, genes, df, agg_fn = rowMeans) {
  df |>
    transmute(
      sampleId,
      !!panel_name := agg_fn(pick(any_of(genes)), na.rm = TRUE)
    )
}

# Mean AdjTPM scores — suitable for log10p1 y-axis in validation figures
gene_sets_tpm <- map2(
  names(gene_panels), gene_panels,
  ~ compute_panel(.x, .y, isofox_wide_tpm)
) |>
  reduce(inner_join, by = 'sampleId') |>
  rename(tcell_tpm = T_Cell)

# Effector T-cell (tGE8) score — mean of raw AdjTPM, same aggregation as
# the T-cell signature above (compute_panel()'s default agg_fn = rowMeans).
gene_sets_effector <- map2(
  names(effector_panels), effector_panels,
  ~ compute_panel(.x, .y, isofox_wide_tpm)
) |>
  reduce(inner_join, by = 'sampleId') |>
  rename(rna_teff_score = Teff_CD8)

# PD-L1 (CD274) score -- single-gene "panel", so the mean below reduces to
# CD274's own raw AdjTPM value; compute_panel() is reused anyway for
# consistency (same join/column-naming pattern as the other scores).
gene_sets_pdl1 <- map2(
  names(pdl1_panel), pdl1_panel,
  ~ compute_panel(.x, .y, isofox_wide_tpm)
) |>
  reduce(inner_join, by = 'sampleId') |>
  rename(rna_pdl1_score = PD_L1)

# T-cell (HCL) score -- mean of raw AdjTPM, same aggregation as the
# primary T-cell signature above (compute_panel()'s default agg_fn =
# rowMeans). Independent, second T-cell presence signature (Han et al.
# 2020) -- see gene panel definition above for citation.
gene_sets_hcl_tcell <- map2(
  names(hcl_tcell_panel), hcl_tcell_panel,
  ~ compute_panel(.x, .y, isofox_wide_tpm)
) |>
  reduce(inner_join, by = 'sampleId') |>
  rename(rna_tcell_hcl_score = HCL_TCell)

# Checkpoint/exhaustion score -- mean of raw AdjTPM, same aggregation as
# every panel above except CYT (see gene panel definition above for the
# panel's canonical, cross-paper status rather than a single citation).
gene_sets_exhaustion <- map2(
  names(exhaustion_panel), exhaustion_panel,
  ~ compute_panel(.x, .y, isofox_wide_tpm)
) |>
  reduce(inner_join, by = 'sampleId') |>
  rename(rna_exhaustion_score = Exhaustion)


# T-cell-inflamed GEP (Ayers et al. 2017) score -- mean of raw AdjTPM, same
# aggregation as every panel above except CYT. See gene panel definition
# above for the verification caveat (secondary-source reconstruction).
gene_sets_ayers_geps <- map2(
  names(ayers_geps_panel), ayers_geps_panel,
  ~ compute_panel(.x, .y, isofox_wide_tpm)
) |>
  reduce(inner_join, by = 'sampleId') |>
  rename(rna_ayers_geps_score = Ayers_GEPS)

gene_sets_ready <- inner_join(gene_sets_tpm, gene_sets_effector, by = 'sampleId') |>
  inner_join(gene_sets_pdl1, by = 'sampleId') |>
  inner_join(gene_sets_hcl_tcell, by = 'sampleId') |>
  inner_join(gene_sets_exhaustion, by = 'sampleId') |>
  inner_join(gene_sets_ayers_geps, by = 'sampleId')

message('Gene sets computed for ', nrow(gene_sets_ready), ' samples')
head(gene_sets_ready)


# 4 - Save output

In [ ]:
# Saves 6 gene-set columns: the mean-AdjTPM T-cell signature (_tpm), the
# mean-AdjTPM CD8 Effector (rna_teff_score), the PD-L1 (rna_pdl1_score), the
# second independent T-cell (HCL) signature (rna_tcell_hcl_score), the
# checkpoint/exhaustion score (rna_exhaustion_score), and the
# T-cell-inflamed GEP score (rna_ayers_geps_score, secondary-source
# reconstruction -- see panel definition above).
fwrite(gene_sets_ready, paste0(CIDER_PROJ_DIR, 'isofox_genesets_ready_log2.csv'))
message('Saved: ', paste0(CIDER_PROJ_DIR, 'isofox_genesets_ready_log2.csv'),
        ' (', ncol(gene_sets_ready) - 1, ' gene set columns)')